In [1]:
from vllm import LLM, EngineArgs
from vllm.utils import FlexibleArgumentParser
from vllm.sampling_params import SamplingParams
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["CUDA_HOME"] = "/usr/local/cuda"
# os.environ["VLLM_PP_LAYER_PARTITION"] = "4,4,4,4,4,4,4,2"
# os.environ["VLLM_PP_LAYER_PARTITION"] = "8,8,8,6"
import torch

INFO 10-17 03:00:50 [__init__.py:244] Automatically detected platform cuda.


In [ ]:
# 提取采样参数
max_tokens = 512
temperature = 0.7
top_p = 0.9
top_k = 50

In [ ]:
# 构建采样对象
sampling_params = SamplingParams(
    max_tokens=max_tokens,
    temperature=temperature,
    top_p=top_p,
    top_k=top_k,
)

In [ ]:
def create_parser():
    parser = FlexibleArgumentParser()
    EngineArgs.add_cli_args(parser)

    # 默认模型路径改成本地 Qwen2.5-7B-Instruct
    parser.set_defaults(model="/workspace/zimoliu/models/Qwen2.5-7B-Instruct")

    # 采样参数（命令行可覆盖）
    sampling_group = parser.add_argument_group("Sampling parameters")
    sampling_group.add_argument("--max-tokens", type=int, default=512)
    sampling_group.add_argument("--temperature", type=float, default=0.7)
    sampling_group.add_argument("--top-p", type=float, default=0.9)
    sampling_group.add_argument("--top-k", type=int, default=50)

    return parser

In [ ]:
def main(args: dict):
    # 提取采样参数
    max_tokens = args.pop("max_tokens")
    temperature = args.pop("temperature")
    top_p = args.pop("top_p")
    top_k = args.pop("top_k")

    # 构建采样对象
    sampling_params = SamplingParams(
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
    )

    # 创建 LLM 实例
    llm = LLM(**args)

    print(">>> Qwen2.5-7B-Instruct 已加载，输入 q 退出对话 <<<")

    while True:
        try:
            user_input = input("\n你：").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n再见！")
            break

        if user_input.lower() == "q":
            print("再见！")
            break
        if not user_input:
            continue

        # 构造单轮对话（无历史）
        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_input},
        ]

        # 调用 vLLM 的 chat 接口
        outputs = llm.chat([conversation], sampling_params, use_tqdm=False)

        # 取第一条结果
        assistant_reply = outputs[0].outputs[0].text.strip()
        print("\n助手：", assistant_reply)

In [ ]:
# if __name__ == "__main__":
parser = create_parser()
args = vars(parser.parse_args())
main(args)

In [ ]:
# Notebook 单 cell 版：Qwen2.5-7B-Instruct 对话
from vllm import LLM
from vllm.sampling_params import SamplingParams



In [ ]:
# ========== 1. 默认参数 ==========
MODEL_PATH = "/workspace/zimoliu/models/Qwen2.5-7B-Instruct"
SAMPLING_KWARGS = dict(
    max_tokens=2048,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
)

# MODEL_PATH = "/sharedata/zimoliu/models/Jamba-v0.1"
# SAMPLING_KWARGS = dict(
#     max_tokens=128,
#     temperature=0.7,
#     top_p=0.9,
#     top_k=50,
# )


In [ ]:
# ========== 2. 加载模型 ==========
print(">>> 正在加载模型，请稍候...")
llm = LLM(
    model=MODEL_PATH,
    # pipeline_parallel_size=8,
    # pipeline_parallel_size=4,
    # 如有其它 EngineArgs，可在此追加，例如：
    # tensor_parallel_size=1,
    # gpu_memory_utilization=0.8,
    # 关键：关闭 flashinfer，用原生 top-k/top-p
    # enforce_eager=True,
    # 或者
    # disable_flashinfer=True,
)
sampling_params = SamplingParams(**SAMPLING_KWARGS)
print(">>> 模型已就绪，输入 q 退出对话 <<<")

In [2]:
# MODEL_PATH = "/sharedata/zimoliu/ckpts/jamba_60B_128k_v6_1node_pp8_ep1_official_ckpt38000/hf"
MODEL_PATH = "/sharedata/zimoliu/ckpts/jamba_60b_aws_oh_pp8_ep4_efa_512k_sft_v1_16node_ckpt75000/hf"
SAMPLING_KWARGS = dict(
    max_tokens=2048,
    temperature=0.75,
    top_p=0.9,
    top_k=1,
)

In [3]:
# ========== 2. 加载模型 ==========
print(">>> 正在加载模型，请稍候...")
llm = LLM(
    model=MODEL_PATH,
    # pipeline_parallel_size=8,
    # pipeline_parallel_size=4,
    # 如有其它 EngineArgs，可在此追加，例如：
    # tensor_parallel_size=1,
    # gpu_memory_utilization=0.8,
    # 关键：关闭 flashinfer，用原生 top-k/top-p
    enforce_eager=True,
    # 或者
    # disable_flashinfer=True,
)
sampling_params = SamplingParams(**SAMPLING_KWARGS)
print(">>> 模型已就绪，输入 q 退出对话 <<<")



>>> 正在加载模型，请稍候...
### llm init kwargs: {'disable_log_stats': True}

### llm init engine_args: EngineArgs(model='/sharedata/zimoliu/ckpts/jamba_60b_aws_oh_pp8_ep4_efa_512k_sft_v1_16node_ckpt75000/hf', served_model_name=None, tokenizer=None, hf_config_path=None, task='auto', skip_tokenizer_init=False, enable_prompt_embeds=False, tokenizer_mode='auto', trust_remote_code=False, allowed_local_media_path='', download_dir=None, load_format='auto', config_format='auto', dtype='auto', kv_cache_dtype='auto', seed=None, max_model_len=None, cuda_graph_sizes=[512], distributed_executor_backend=None, pipeline_parallel_size=1, tensor_parallel_size=1, data_parallel_size=1, data_parallel_rank=None, data_parallel_size_local=None, data_parallel_address=None, data_parallel_rpc_port=None, data_parallel_backend='mp', enable_expert_parallel=False, enable_eplb=False, num_redundant_experts=0, eplb_window_size=1000, eplb_step_interval=3000, eplb_log_balancedness=False, max_parallel_loading_workers=None, block_s

You are using a model of type jambadoe to instantiate a model of type jamba_doe. This is not supported for all configurations of models and can yield errors.


### get_config config_dict: {'architectures': ['JambaDoEForCausalLM'], 'attention_dropout': 0.0, 'bos_token_id': 151643, 'eos_token_id': 151645, 'hidden_act': 'silu', 'hidden_size': 4096, 'initializer_range': 0.02, 'intermediate_size': 8192, 'max_position_embeddings': 32768, 'rope_scaling': {'factor': 40, 'mscale': 0.707, 'mscale_all_dim': 0.707, 'beta_fast': 32, 'beta_slow': 1, 'original_max_position_embeddings': 524288, 'extrapolation_factor': 1.0, 'rope_type': 'deepseek_yarn'}, 'max_window_layers': 30, 'model_type': 'jambadoe', 'num_attention_heads': 32, 'num_hidden_layers': 30, 'num_key_value_heads': 32, 'rms_norm_eps': 1e-05, 'rope_theta': 100000000.0, 'tie_word_embeddings': False, 'torch_dtype': 'bfloat16', 'transformers_version': '4.43.1', 'use_cache': True, 'use_sliding_window': False, 'vocab_size': 151680, '_commit_hash': None}
### rope scaling is None, init with default value.

### rope scaling is None, init with default value.

### hf_config: JambaDoEConfig {
  "architecture

Loading safetensors checkpoint shards:   0% Completed | 0/30 [00:00<?, ?it/s]


INFO 10-17 03:05:18 [default_loader.py:278] Loading weights took 246.91 seconds
### rank: 0, model_config.quantization: None, loaded_weights: {'model.layers.8.mamba.conv1d.weight', 'model.layers.25.mlp.experts.w13_weight', 'model.layers.28.mlp.shared_experts_gate.weight', 'model.layers.9.mamba.out_proj.weight', 'model.layers.26.mamba.conv1d.weight', 'model.layers.9.input_layernorm.weight', 'model.layers.16.mlp.shared_experts_gate.weight', 'model.layers.4.mlp.gate.weight', 'model.layers.14.pre_ff_layernorm.weight', 'model.layers.11.self_attn.kv_b_proj.weight', 'model.layers.0.mlp.shared_experts.gate_up_proj.weight', 'model.layers.4.mamba.dt_bias', 'model.layers.12.input_layernorm.weight', 'model.layers.18.mamba.norm.weight', 'model.layers.14.mamba.conv1d.bias', 'model.layers.19.mlp.gate.weight', 'model.layers.13.mlp.shared_experts_gate.weight', 'model.layers.27.knowledge_attn.kn_up_proj.weight', 'model.layers.20.mlp.experts.w2_weight', 'model.layers.3.self_attn.kv_b_proj.weight', 'model

In [ ]:
print(123)

In [4]:
def chat_loop():
    while True:
        try:
            user_input = input("").strip()          # 去掉提示符
        except (KeyboardInterrupt, EOFError):
            print("\n再见！")
            break
        if user_input.lower() == "q":
            print("再见！")
            break
        if not user_input:
            continue

        # 打印用户输入
        print(f"\n你：{user_input}")

        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_input},
        ]
        # outputs = llm.generate(prompts=user_input, sampling_params=sampling_params)
        outputs = llm.chat([conversation], sampling_params, use_tqdm=True)
        assistant_reply = outputs[0].outputs[0].text.strip()
        print(f"助手：{assistant_reply}")

In [5]:
# 运行对话
chat_loop()


你：我去早市买东西，买了一斤苹果，2.5千克香蕉，苹果是3元每斤，香蕉是2元每斤，我一共要花多少钱？注意单位换算
INFO 10-17 03:11:46 [chat_utils.py:444] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
### prompt_str: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
我去早市买东西，买了一斤苹果，2.5千克香蕉，苹果是3元每斤，香蕉是2元每斤，我一共要花多少钱？注意单位换算<|im_end|>
<|im_start|>assistant




Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

助手：我们来逐步计算这个问题。

### 已知条件：
1. 苹果的价格是 **3元/斤**，买了 **1斤**。
2. 香蕉的价格是 **2元/斤**，买了 **2.5千克**。
3. 需要将香蕉的重量从千克转换为斤，因为1千克 = 2斤。

---

### 第一步：计算香蕉的重量（单位换算）
香蕉的重量是 **2.5千克**，换算成斤：
$$
2.5 \, \text{千克} \times 2 = 5 \, \text{斤}
$$

---

### 第二步：计算苹果的总价
苹果的价格是 **3元/斤**，买了 **1斤**：
$$
1 \, \text{斤} \times 3 \, \text{元/斤} = 3 \, \text{元}
$$

---

### 第三步：计算香蕉的总价
香蕉的价格是 **2元/斤**，买了 **5斤**：
$$
5 \, \text{斤} \times 2 \, \text{元/斤} = 10 \, \text{元}
$$

---

### 第四步：计算总价
将苹果和香蕉的总价相加：
$$
3 \, \text{元} + 10 \, \text{元} = 13 \, \text{元}
$$

---

### 最终答案：
$$
\boxed{13 \, \text{元}}
$$
再见！


我们来逐步计算这个问题。

### 已知条件：
1. 苹果的价格是 **3元/斤**，买了 **1斤**。
2. 香蕉的价格是 **2元/斤**，买了 **2.5千克**。
3. 需要将香蕉的重量从千克转换为斤，因为1千克 = 2斤。

---

### 第一步：计算香蕉的重量（单位换算）
香蕉的重量是 **2.5千克**，换算成斤：
$$
2.5 \, \text{千克} \times 2 = 5 \, \text{斤}
$$

---

### 第二步：计算苹果的总价
苹果的价格是 **3元/斤**，买了 **1斤**：
$$
1 \, \text{斤} \times 3 \, \text{元/斤} = 3 \, \text{元}
$$

---

### 第三步：计算香蕉的总价
香蕉的价格是 **2元/斤**，买了 **5斤**：
$$
5 \, \text{斤} \times 2 \, \text{元/斤} = 10 \, \text{元}
$$

---

### 第四步：计算总价
将苹果和香蕉的总价相加：
$$
3 \, \text{元} + 10 \, \text{元} = 13 \, \text{元}
$$

---

### 最终答案：
$$
\boxed{13 \, \text{元}}
$$

In [ ]:
# 运行对话
chat_loop()
del llm
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [ ]:
# 运行对话
chat_loop()
del llm
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [ ]:
del llm
torch.cuda.empty_cache()
torch.cuda.synchronize()